# Data Understanding

This notebook loads the EMSCAD (Employment Scam Aegean Dataset) job postings dataset and confirms its basic shape, class balance, missing-value pattern, presence of duplicate records, and a first look at the text and metadata fields. The goal of this notebook is to understand the dataset being used, not to clean or model the data yet.

In [1]:
# Import pandas and configure it to display every column, not a truncated subset
import pandas as pd
pd.set_option("display.max_columns", None)

## Load the Dataset

The EMSCAD CSV file is loaded into a dataframe, and its shape is printed. This confirms the number of rows and columns available to work with.

In [ ]:
# Load the job postings dataset and report its shape
df = pd.read_csv("../data/raw/fake_job_postings.csv")
print(df.shape)

The dataset contains 17,880 rows and 18 columns.

## Preview the Data

Knowing the shape alone is not enough to understand the dataset. Seeing the actual row content, the dtype pandas has assigned to each column, and summary statistics for the numeric columns is also important to ensure that later work (text cleaning, metadata handling) is based on real data.

In [3]:
# Preview the first 5 rows to see actual field content
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


The text fields (`company_profile`, `description`, `requirements`, `benefits`) contain long lines of text, including raw formatting artifacts such as encoding glitches. This confirms that text cleaning is necessary. `location` is a single combined string (country, state, city), and `salary_range` contains null values which confirms that missing values need to be handled later on.

In [4]:
# Check each column's dtype and non-null count in one summary
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_id               17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17880 non-null  int64
 10  has_company_logo     17880 non-null  int64
 11  has_questions        17880 non-null  int64
 12  employment_type      14409 non-null  str  
 13  required_experience  10830 non-null  str  
 14  required_education   9775 non-null   str  
 15  industry             12977 non-null  str  
 16  function             11425 non-nu

All 18 columns are loaded with the expected dtype. `job_id`, `telecommuting`, `has_company_logo`, `has_questions`, and `fraudulent` are integers, and every other column is text.

In [5]:
# Summarise the numeric columns
df.describe()

,job_id,telecommuting,has_company_logo,has_questions,fraudulent
count,17880.000000,17880.000000,17880.000000,17880.000000,17880.000000
mean,8940.500000,0.042897,0.795302,0.491723,0.048434
std,5161.655742,0.202631,0.403492,0.499945,0.214688
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,4470.750000,0.000000,1.000000,0.000000,0.000000
50%,8940.500000,0.000000,1.000000,0.000000,0.000000
75%,13410.250000,0.000000,1.000000,1.000000,0.000000
max,17880.000000,1.000000,1.000000,1.000000,1.000000


`describe()` only summarises the five numeric columns since every other column contains text. `job_id` is just a row identifier and carries no modelling signal. The three binary flag columns are numeric, so their averages double as simple rates: only 4.29% of postings are telecommuting, 79.53% have a company logo, and 49.17% include screening questions.

## Check the Class Balance

The exact ratio of fraudulent versus legitimate postings is confirmed to ensure that every later modelling decision (e.g. class weighting, choice of metric) is justified by a real number instead of an assumption.

In [6]:
# Calculate the proportion of fraudulent versus legitimate postings
fraud_rate = df["fraudulent"].value_counts(normalize=True).apply(lambda x: f"{x * 100:.2f}%")
print(fraud_rate)

fraudulent
0    95.16%
1     4.84%
Name: proportion, dtype: str


Fraudulent postings make up approximately 4.8% of the dataset. This confirms that accuracy alone will be a misleading evaluation metric later on, because a model that always identifies a post as legitimate will be correct 95% of the time.

## Inspect Missing Values

Columns like `salary_range` are known to contain missing values. The number and percentage of missing values per column are checked to gain insights that inform how missing values are handled later.

In [7]:
# Calculate the number of missing values per column, sorted from most to least missing
missing_values_by_column = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
print("Count:")
print(missing_values_by_column)

# Calculate the percentage of missing values per column, sorted from most to least missing
missing_value_percentage_by_column = (df.isna().mean() * 100).round(2)
missing_value_percentage_by_column = missing_value_percentage_by_column[missing_value_percentage_by_column > 0].sort_values(ascending=False).apply(lambda x: f"{x:.2f}%")
print("\nPercentage:")
print(missing_value_percentage_by_column)

Count:
salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
dtype: int64

Percentage:
salary_range           83.96%
department             64.58%
required_education     45.33%
benefits               40.34%
required_experience    39.43%
function               36.10%
industry               27.42%
employment_type        19.41%
company_profile        18.50%
requirements           15.08%
location                1.94%
description             0.01%
dtype: str


`salary_range` is missing in 83.96% of rows, leaving too few real values (only about 16% of postings) to use as a normal feature on its own. `department` (64.58%), `required_education` (45.33%), `benefits` (40.34%), `required_experience` (39.43%), `function` (36.10%), `industry` (27.42%), `employment_type` (19.41%), `company_profile` (18.50%), and `requirements` (15.08%) all have meaningful missing data and will need a placeholder value filled in rather than being dropped outright. `location` is only missing in 1.94% of rows, and `description`, `title`, `job_id`, `telecommuting`, `has_questions`, `has_company_logo`, and `fraudulent` are effectively complete (0% or near-0% missing).

## Look at the Text Fields

The text fields will be combined and cleaned in `02_data_preparation.ipynb`, so a closer look at how much raw content they actually carry is useful now. The character length of each field is measured (treating a missing value as length 0), and one legitimate and one fraudulent posting are printed side by side to see what the raw text looks like before any cleaning is applied.

In [ ]:
# Calculate the character length of each text field, treating a missing value as length 0
text_fields = ["title", "company_profile", "description", "requirements", "benefits"]
text_lengths = df[text_fields].fillna("").apply(lambda column: column.str.len())
text_lengths.describe().round(1)

`description` is the longest field on average (about 1,218 characters) and the most variable (up to nearly 15,000 characters), while `benefits` is the shortest and most often empty (median 45 characters, with a quarter of postings having none at all). This wide spread confirms that combining the text fields into one blob, as planned for `02_data_preparation.ipynb`, needs to tolerate very short and very long postings alike.

In [ ]:
# Print the description of one legitimate and one fraudulent posting to see raw text firsthand
legitimate_example = df.loc[df["fraudulent"] == 0, "description"].dropna().iloc[0]
fraudulent_example = df.loc[df["fraudulent"] == 1, "description"].dropna().iloc[0]

print("Legitimate posting example:\n", legitimate_example[:300])
print("\nFraudulent posting example:\n", fraudulent_example[:300])

Both examples read as ordinary job-description prose, but the fraudulent example contains a raw HTML entity (`&amp;`) and a stray encoding artifact (`�`), consistent with the formatting glitches already noted in the earlier preview. A single example from each class is not proof of a systematic pattern, but it does confirm that the text cleaning step in `02_data_preparation.ipynb` needs to handle HTML entities and encoding artifacts, not just ordinary punctuation.

## Look at the Metadata Fields

The low-cardinality categorical fields and the binary flag columns are looked at next: their most common categories, and whether the fraud rate visibly differs across their values. This is checked because a field that splits cleanly by class would be a strong, cheap signal for the classical model, while a field with no visible split may contribute little relative to the missing-value burden it carries.

In [ ]:
# Show the category counts for the low-cardinality metadata fields
categorical_fields = ["employment_type", "required_experience", "required_education"]
for field in categorical_fields:
    print(df[field].value_counts())
    print()

Full-time (11,620 postings), Mid-Senior level (3,809), and Bachelor's Degree (5,145) are each the dominant category in their column, so these fields lean toward one value but are not reduced to it — every field still has a meaningful spread across its remaining categories.

In [ ]:
# Compare the fraud rate across each binary flag column
binary_fields = ["telecommuting", "has_company_logo", "has_questions"]
for field in binary_fields:
    print((df.groupby(field)["fraudulent"].mean() * 100).round(2))
    print()

All three binary flags show a visible split: telecommuting postings are fraudulent almost twice as often (8.34% vs 4.69%), postings without a company logo are fraudulent nearly eight times as often (15.93% vs 1.99%), and postings without screening questions are fraudulent more than twice as often (6.78% vs 2.84%). `has_company_logo` in particular looks like a strong, complete (0% missing) signal worth keeping as a feature in the classical model.

## Check for Duplicate Records

Duplicate postings can leak between the train, validation, and test splits later on, letting the model be evaluated on something it has already partly memorised, resulting in the inflation of reported metrics. The dataset is checked for exact full-row duplicates, as well as duplicate `job_id` values, which should be unique by construction.

In [8]:
# Count exact full-row duplicates and duplicate job_id values
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate job_id values:", df["job_id"].duplicated().sum())

Full-row duplicates: 0
Duplicate job_id values: 0


No full-row duplicates and no duplicate `job_id` values were found. This dataset does not need de-duplication before the train/validation/test split.

## Summary

- Confirmed the dataset loads to 17,880 rows and 18 columns.
- Confirmed the fraud rate is approximately 4.8% (866 fraudulent postings), justifying `class_weight="balanced"` over resampling (e.g. SMOTE) later on. `class_weight="balanced"` penalises misclassifying the minority (fraudulent) class more heavily during training without changing the training data itself, whereas SMOTE would generate synthetic fraudulent rows by interpolating in feature space. Since the classical model's features are high-dimensional, sparse TF-IDF vectors, interpolating between them is both computationally expensive and prone to producing synthetic feature vectors that do not correspond to any real, coherent text. Hence, `class_weight="balanced"` is the more effective choice.
- Confirmed which columns have heavy missing data (`salary_range`, `department`, `required_education`, `benefits`, `required_experience`, `function`, `industry`, `employment_type`, `company_profile`, `requirements`).
- Confirmed the text fields vary widely in length and carry raw HTML/encoding artifacts, confirming that both combining and cleaning are needed before modelling.
- Confirmed the binary flag columns (`telecommuting`, `has_company_logo`, `has_questions`) each show a visible fraud-rate split, with `has_company_logo` the strongest and most complete of the three — worth keeping as a feature in the classical model.
- Confirmed there are no full-row duplicates or duplicate `job_id` values, so no de-duplication is needed before the train/validation/test split.
- No data was cleaned or transformed in this notebook. That begins in `02_data_preparation.ipynb`.